# Deep Research Agent с использованием OpenAI Agents SDK

В этом ноутбуке мы создаём **Deep Research Agent** — агентa с многошаговым рассуждением, который использует **agentic loop** для глубокого исследования тем.

Библиотека [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) - это библиотека для построение агентов и многоагентных систем на основе моделей, поддерживающих Responses API. Она позволяет создавать как и отдельных агентов (наподобие определённого нами класса `Agent`), так и группы работающих совместно агентов.

Мы рассмотрим несколько возможных архитектур для проведения исследований.

## Агентский цикл

Основное отличие полноценного агента (или т.н. **ReAct-агента**, от Reasoning and Acting) - это наличие **агентского цикла**, в котором:

1. Агент получает задачу
1. Планирует, на какие подзадачи можно её разбить
1. Решает, какой инструмент использовать
1. Выполняет действие
1. Анализирует результат
1. Повторяет, пока задача не выполнена

В нашем случае в качестве инструмента агент будет использовать инструмент сохранения заметок. Идея в том, что в процессе поиска в интернет агент будет сохранять важные заметки, а затем формировать итоговый отчёт на основе этих материалов.

```
┌────────────────────────────────────────────────────────────────┐
│                    OpenAI Agents SDK                           │
│                                                                │
│                        ┌─────────┐                             │
│                        │  Agent  │                             │
│                        └─────────┘                             │
│                ┌────────────┴────────────┐                     │
│                │                         │                     │
│                ▼                         ▼                     │
│         ┌────────────┐            ┌────────────┐               │
│         │ Web Search │            │   Notes    │               │
│         │   Tool     │            │    Tool    │               │
│         └────────────┘            └────────────┘               │
└────────────────────────────────────────────────────────────────┘
                          │
                          ▼
                    Yandex Cloud
                    Responses API
```

Установим необходимые библиотеки:

In [ ]:
%pip install --quiet openai-agents python-dotenv

## Авторизация

Для работы с Yandex Cloud нам понадобится `folder_id` (идентификатор каталога) и `api_key` (API-ключ сервисного аккаунта). Мы предполагаем, что эти значения хранятся в файле `.env` в текущей директории.

In [ ]:
!curl -o .env {{url_of_dotenv_file}}

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

def printx(string):
    display(Markdown(string))

print(f"✅ Авторизация настроена (folder_id: {folder_id[:8]}...)")

✅ Авторизация настроена (folder_id: b1gbicod...)


---

## Часть 1: Настройка OpenAI Agents SDK для работы с Yandex Cloud

OpenAI Agents SDK позволяет использовать асинхронное взаимодействие с моделью - это значит, что во время ожидания ответа модели выполнение локального цикла не блокируется, что повышает отзывчивость системы. Более того, в перспективе можно строить ответы на основе **стриминга**, когда ответ начинает поступать постепенно по мере генерации.

Для поддержки асинхронности используется клиент `AsyncOpenAI`. Создаётся он таким же образом, как и ранее используемый нами клиент `OpenAI`.

Для работы Agents SDK мы дополнительно определяем объект `OpenAIResponsesModel`, который в себе инкапсулирует OpenAI-клиента и название используемой модели (в нашем случае Qwen3): 

In [ ]:
from agents import Agent, Runner, WebSearchTool, function_tool, set_tracing_disabled
from agents.models.openai_responses import OpenAIResponsesModel
from openai import AsyncOpenAI

# Отключаем трейсинг (он требует OpenAI API key)
set_tracing_disabled(True)

# Создаём клиент для Yandex Cloud
yandex_client = AsyncOpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id
)

# Модель для использования
model_uri = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"

# Создаём модель для Agents SDK
yandex_model = OpenAIResponsesModel(
    model=model_uri,
    openai_client=yandex_client
)

print(f"✅ Модель настроена: {model_uri}")

✅ Модель настроена: gpt://b1gbicod0scglhd49qs0/qwen3-235b-a22b-fp8/latest


---

## Часть 2: Создание инструментов для исследования

Определим функции инструмента ведения заметок. В нашем примере будем использовать обычный спискок заметок, хотя в более сложном случае вы можете захотеть размещать найденные заметки в базе данных или в семантическом хранилище, чтобы потом иметь возможность осуществлять беседу с агентом на заданную тему. Также возможно имеет смысл привязывать заметки к теме исследования.

In [ ]:
# Инструмент для сохранения заметок
research_notes = []

@function_tool
def save_note(topic: str, content: str) -> str:
    """
    Сохраняет заметку в базу исследования.
    
    Args:
        topic: Тема заметки
        content: Содержание заметки
    
    Returns:
        Подтверждение сохранения
    """
    research_notes.append({"topic": topic, "content": content})
    return f"✅ Заметка '{topic}' сохранена ({len(research_notes)} всего)"


@function_tool
def get_all_notes() -> str:
    """
    Возвращает все сохранённые заметки.
    
    Returns:
        Список всех заметок
    """
    if not research_notes:
        return "Заметок пока нет"
    
    result = "📝 Сохранённые заметки:\n\n"
    for i, note in enumerate(research_notes, 1):
        result += f"{i}. **{note['topic']}**\n{note['content'][:200]}...\n\n"
    return result


@function_tool
def clear_notes() -> str:
    """
    Очищает все заметки для начала нового исследования.
    
    Returns:
        Подтверждение очистки
    """
    global research_notes
    count = len(research_notes)
    research_notes = []
    return f"🗑️ Очищено {count} заметок"


print("✅ Инструменты для заметок созданы")

✅ Инструменты для заметок созданы


Обратите внимание, что создание инструментов - очень простая задача, мы просто декорируем Python-функцию с помощью `@function_tool`, и описываем её предназначение и параметры в виде docstring-документации и с помощью типизации.

---

## Часть 3: Создание Deep Research Agent

Простейший Deep Research агент - это LLM с инструментами:
- **WebSearchTool** для поиска в интернете
- Локальными функциями для управления заметками

Также нам нужно описать в системном промпте логику работы LLM.

In [ ]:
deep_research_instructions = """
Ты — Deep Research Agent, эксперт по глубокому исследованию тем.

## Твоя методология:

1. **Планирование**: Разбей тему на 3-5 ключевых аспектов для исследования
2. **Поиск**: Для каждого аспекта выполни поиск в интернете
3. **Анализ**: Извлеки ключевые факты и сохрани заметки
4. **Синтез**: Объедини найденную информацию в связный отчёт

## Правила:

- Всегда начинай с планирования исследования
- Делай несколько поисковых запросов для полноты картины
- Сохраняй важные находки с помощью save_note
- Указывай источники информации
- В финальном отчёте структурируй информацию логично

## Формат финального отчёта:

# [Название темы]

## Резюме
[Краткое описание 2-3 предложения]

## Ключевые находки
[Основные факты и тренды]

## Детальный анализ
[Глубокий разбор по аспектам]

## Выводы
[Заключения и рекомендации]

Отвечай на русском языке.
"""

# Создаём Deep Research Agent
deep_research_agent = Agent(
    name="DeepResearchAgent",
    model=yandex_model,
    instructions=deep_research_instructions,
    tools=[
        WebSearchTool(),  # Поиск в интернете
        save_note,        # Сохранение заметок
        get_all_notes,    # Получение заметок
        clear_notes,      # Очистка заметок
    ]
)

print("🔬 Deep Research Agent создан!")

🔬 Deep Research Agent создан!


---

## Часть 4: Запуск исследования

Для запуска агента используется специальный объект `Runner`. `Runner.run` выполняет **агентский цикл**, в котором агент:
1. Получает задачу
2. Выбирает инструмент
3. Выполняет действие
4. Анализирует результат
5. Решает, нужны ли ещё действия

In [ ]:
async def deep_research(topic: str, verbose: bool = True):
    """
    Выполняет глубокое исследование темы.
    
    Args:
        topic: Тема для исследования
        verbose: Выводить ли прогресс
    
    Returns:
        Результат исследования
    """
    # Очищаем предыдущие заметки
    global research_notes
    research_notes = []
    
    if verbose:
        print(f"🔬 Начинаем исследование: {topic}")
        print("=" * 60)
    
    # Запускаем agentic loop
    result = await Runner.run(
        deep_research_agent,
        f"Проведи глубокое исследование темы: {topic}"
    )
    
    if verbose:
        print("=" * 60)
        print(f"✅ Исследование завершено")
        print(f"📝 Сохранено заметок: {len(research_notes)}")
    
    return result


print("✅ Функция deep_research готова")

✅ Функция deep_research готова


Попробует провести исследование про последние новости в мире агентов:

In [ ]:
# Исследуем тему AI-агентов в 2025 году
result = await deep_research(
    "Развитие AI-агентов в 2025 году: ключевые продукты, компании и тренды"
)

printx(result.final_output)

## Мониторинг (RunHooks)

Исследование занимает достаточно много времени, и нам было бы полезно понимать, какие действия выполняет агент прежде, чем выдать финальный результат. Для этого можно использовать механизм **hooks** — коллбэков, которые вызываются при наступлении тех или иных событий.

Создадим класс `VerboseHooks` для отслеживания действий агента:

In [ ]:
from agents import RunHooks, RunContextWrapper, Tool
from agents.lifecycle import AgentHooks

class VerboseHooks(RunHooks):
    """
    Hooks для вывода действий агента в реальном времени.
    Показывает старт/завершение агентов, вызовы инструментов, hosted tools и handoffs.
    """
    
    async def on_agent_start(self, context: RunContextWrapper, agent):
        print(f"🤖 Агент запущен: {agent.name}")
    
    async def on_agent_end(self, context: RunContextWrapper, agent, output):
        output_preview = str(output)[:100] + "..." if len(str(output)) > 100 else str(output)
        print(f"✅ Агент завершён: {agent.name}")
        print(f"   └─ Результат: {output_preview}")
    
    async def on_tool_start(self, context: RunContextWrapper, agent, tool: Tool):
        print(f"🔧 Вызов инструмента: {tool.name}")
    
    async def on_tool_end(self, context: RunContextWrapper, agent, tool: Tool, result):
        result_preview = str(result)[:80] + "..." if len(str(result)) > 80 else str(result)
        print(f"   └─ Результат: {result_preview}")

    async def on_handoff(self, context: RunContextWrapper, from_agent, to_agent):
        print(f"🔀 Handoff: {from_agent.name} → {to_agent.name}")
    
    async def on_llm_end(self, context: RunContextWrapper, agent, response):
        """Показывает вызовы hosted tools (web_search) и function calls."""
        for item in response.output:
            item_type = getattr(item, 'type', None)
            if item_type == 'web_search_call':
                print(item)
                query = getattr(item, 'action', '')
                query = getattr(query,'query', '')
                query = getattr(query,'query', '')
                print(f"🌐 Web Search: {query}")
            #elif item_type == 'function_call':
            #    print(item)
            #    name = getattr(item, 'name', 'unknown')
            #    args = getattr(item, 'arguments', '')[:50]
            #    print(f"📞 Function call: {name}({args}...)")

# Создаём экземпляр для использования
verbose_hooks = VerboseHooks()

print("✅ VerboseHooks готовы")

✅ VerboseHooks готовы


Теперь запустим исследование с мониторингом:

In [ ]:
# Исследование С мониторингом через hooks
research_notes = []  # Очищаем заметки

print("🔬 Начинаем исследование с мониторингом...")
print("=" * 60)

result = await Runner.run(
    deep_research_agent,
    "Исследуй тему: Новые AI-фреймворки в 2025 году",
    hooks=verbose_hooks  # <-- Добавляем hooks!
)

print("=" * 60)
print(f"📝 Сохранено заметок: {len(research_notes)}")
printx(result.final_output)

Можем посмотреть на то, какие заметки были сохранены:

In [ ]:
# Посмотрим сохранённые заметки
print("📝 Сохранённые заметки:\n")
for i, note in enumerate(research_notes, 1):
    print(f"{i}. {note['topic']}")
    print(f"   {note['content'][:100]}...\n")

---

## Часть 5: Многоагентные паттерны координации

Для решения сложных задач часто удобно использовать множество взаимодействующих агентов - т.н. **многоагентные системы**. Есть несколько причин, по которым одного агента может быть недостаточно:

* При наличии большого количества инструментов (10+), LLM начинает путаться при их вызове, точность вызова инструментов падает. При этом разбивая систему на различные специализированные агенты мы уменьшаем количество инструментов у каждого из них.
* LLM может смотреть на задачу с разных точек зрения в зависимости от системного промпта. Мы можем использовать разные агенты для разных задач, например, при медицинской диагностике сделать координирующего агента-терапевта, который будет передавать случай на рассмотрение более специализированным агентам-врачам по разным специальностям.
* Как и в реальной жизни, если мы добавляем ещё один уровень разбиения сложной задачи на подзадачи, то мы упрощаем задачу для каждого из агентов, что повышает качество решения.

В случае Deep Research мы можем разбить задачу на следующих агентов:

```
┌────────────────────────────────────────────────────────────────┐
│                    OpenAI Agents SDK                          │
│  ┌─────────┐    ┌─────────┐    ┌─────────┐                    │
│  │ Planner │───▶│ Searcher│───▶│ Writer  │                    │
│  │  Agent  │    │  Agent  │    │  Agent  │                    │
│  └─────────┘    └─────────┘    └─────────┘                    │
│       │              │              │                         │
│       ▼              ▼              ▼                         │
│  [Think & Plan] [Web Search]  [Synthesize]                    │
└────────────────────────────────────────────────────────────────┘
                          │
                          ▼
                    Yandex Cloud
                    Responses API
```

Однако возникает вопрос: как эти агенты будут взаимодействовать между собой? Будет ли у них какой-то координатор? Или мы запрограммируем какой-то фиксированный алгоритм взаимодействия?

В OpenAI Agents SDK есть несколько способов организовать взаимодействие агентов:

| Паттерн | Контроль | История | Гибкость | Когда использовать |
|---------|----------|---------|----------|--------------------|
| **Sequential Chain** | Последний агент | Полная | Средняя | Конвейеры |
| **Hub-and-Spoke** | Координатор (LLM) | Полная | Высокая | Динамические workflow |
| **Agents-as-Tools** | Координатор (LLM) | Нет (новый контекст) | Высокая | Сложная оркестрация |

Рассмотрим каждый паттерн на примере исследовательской команды:
- **Planner** — планирует исследование
- **Searcher** — выполняет поиск
- **Writer** — пишет финальный отчёт

### 5.1 Паттерн B: Sequential Chain (Последовательная цепочка)

В этом паттерне агенты образуют цепочку: каждый передаёт управление следующему.

```
User → Planner → Searcher → Writer → User
```

**Особенность:** Агент, который запускает диалог, не получает контроль обратно — финальный агент отвечает пользователю. При этом вся история диалога передаётся между агентами, они видят всю переписку.

In [ ]:
# Агенты определяются в обратном порядке (от конца цепочки к началу)

# Writer — конец цепочки, отвечает пользователю
writer_agent = Agent(
    name="WriterAgent",
    model=yandex_model,
    instructions="""
    Ты — аналитик-писатель. Твоя задача:
    1. Получить результаты поиска
    2. Использовать get_all_notes для сбора информации
    3. Написать финальный аналитический отчёт
    
    Ты — ПОСЛЕДНИЙ в цепочке, твой ответ идёт пользователю.
    Пиши на русском языке.
    """,
    tools=[get_all_notes]
    # Нет handoffs — конец цепочки
)

# Searcher → Writer
searcher_agent = Agent(
    name="SearcherAgent",
    model=yandex_model,
    instructions="""
    Ты — исследователь. Твоя задача:
    1. Получить план исследования
    2. Для каждого вопроса выполнить поиск в интернете
    3. Сохранить ключевые факты с помощью save_note
    4. После завершения поиска — ПЕРЕДАТЬ управление WriterAgent
    """,
    tools=[WebSearchTool(), save_note],
    handoffs=[writer_agent]  # Передаём Writer
)

# Planner → Searcher
planner_agent = Agent(
    name="PlannerAgent",
    model=yandex_model,
    instructions="""
    Ты — планировщик исследований. Твоя задача:
    1. Получить тему исследования
    2. Разбить на 3-5 ключевых вопросов
    3. После составления плана — ПЕРЕДАТЬ управление SearcherAgent
    
    Формат плана:
    ## План исследования
    1. [Вопрос 1]
    2. [Вопрос 2]
    ...
    """,
    handoffs=[searcher_agent]  # Передаём Searcher
)

print("✅ Sequential Pattern: Planner → Searcher → Writer")

✅ Sequential Pattern: Planner → Searcher → Writer


Мы здесь использовали механизм передачи управления между агентами - **handoff**. Следует отметить, что handoff похож на вызов инструментов, поскольку LLM решает, нужно ли передавать управление другому агенту, и если да - то какому (хотя в нашем случае такого вопроса не возникает, т.к. для каждого агента явно прописан следующий, которому можно передать управление).

Посмотрим, как это работает:

In [ ]:
# Запуск Sequential Chain
research_notes = []  # Очищаем заметки

result = await Runner.run(
    planner_agent,
    "Исследуй тему: Как компании используют AI-агентов для автоматизации в 2025 году",
    hooks=verbose_hooks
)

print(f"Финальный агент: {result.last_agent.name}")
printx(result.final_output)

### Стриминг событий (альтернатива Hooks)

Помимо hooks, OpenAI Agents SDK предоставляет ещё один способ мониторинга работы агента — **стриминг событий**. В отличие от hooks, которые требуют определения отдельного класса с коллбэками, стриминг позволяет обрабатывать события в едином асинхронном цикле `async for`.

Для использования стриминга вместо `Runner.run()` вызываем `Runner.run_streamed()`, который возвращает объект `RunResultStreaming`. Метод `result.stream_events()` даёт нам асинхронный поток объектов `StreamEvent`.

Существует три типа событий:

| Тип события | Описание | Пример использования |
|---|---|---|
| `raw_response_event` | Низкоуровневые события от LLM (токен за токеном) | Отображение текста в реальном времени, как в Deep Research от OpenAI |
| `run_item_stream_event` | Высокоуровневые события: вызов инструмента, результат инструмента, сообщение | Отслеживание прогресса: "Вызван инструмент X", "Результат: Y" |
| `agent_updated_stream_event` | Смена текущего агента (handoff) | Отображение передачи управления между агентами |

Стриминг особенно полезен для отображения **генерации текста в реальном времени** — мы видим ответ агента по мере его создания, а не ждём полного завершения.

Давайте запустим тот же Sequential Chain, но теперь с использованием стриминга:

In [ ]:
from openai.types.responses import ResponseTextDeltaEvent
from agents import ItemHelpers

# Запуск Sequential Chain со стримингом
research_notes = []  # Очищаем заметки

print("🔬 Запуск Sequential Chain со стримингом...")
print("=" * 60)

result = Runner.run_streamed(
    planner_agent,
    "Исследуй тему: Как компании используют AI-агентов для автоматизации в 2025 году"
)

async for event in result.stream_events():
    # Смена агента (handoff)
    if event.type == "agent_updated_stream_event":
        print(f"\n🔀 Агент: {event.new_agent.name}")
        print("-" * 40)

    # Высокоуровневые события: вызовы инструментов и их результаты
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            if event.item.raw_item.type == 'web_search_call':
                print(f"🔧 Вызов web search, query={event.item.raw_item.action.query}")
            else:
                print(f"🔧 Вызов инструмента: {event.item.raw_item.name}")
        elif event.item.type == "tool_call_output_item":
            output_preview = str(event.item.output)[:100]
            print(f"   └─ Результат: {output_preview}...")
        elif event.item.type == "message_output_item":
            # Финальное сообщение уже было отображено токен за токеном
            pass

    # Потоковый вывод текста от LLM (токен за токеном)
    elif event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

print("\n" + "=" * 60)
print(f"📝 Сохранено заметок: {len(research_notes)}")
print(f"✅ Финальный агент: {result.last_agent.name}")

🔬 Запуск Sequential Chain со стримингом...

🔀 Агент: PlannerAgent
----------------------------------------
## План исследования
1. Какие типы AI-агентов наиболее распространены в бизнесе в 2025 году?
2. В каких областях (продажи, поддержка, логистика и т.д.) компании активнее всего внедряют AI-агентов?
3. Какие технологические и организационные барьеры возникают при автоматизации с помощью AI-агентов?
4. Какие примеры успешного внедрения AI-агентов в крупных компаниях в 2025 году?
5. Каковы прогнозируемые тенденции развития использования AI-агентов в ближайшие годы?

🔧 Вызов инструмента: transfer_to_searcheragent

🔀 Агент: SearcherAgent
----------------------------------------
🔧 Вызов web search, query={"query": "Какие типы AI-агентов наиболее распространены в бизнесе в 2025 году", "lang": "ru"}
🔧 Вызов инструмента: save_note
   └─ Результат: ✅ Заметка 'Типы AI-агентов' сохранена (2 всего)...
🔧 Вызов web search, query={"query": "области внедрения AI-агентов в бизнесе 2025", "lang": "ru

Обратите внимание на разницу между двумя подходами:

| Аспект | Hooks | Стриминг |
|--------|-------|----------|
| **Механизм** | Класс с коллбэками | Асинхронный цикл `async for` |
| **Текст в реальном времени** | ❌ Только после завершения | ✅ Токен за токеном |
| **Уровень детализации** | Высокий (отдельные методы) | Гибкий (фильтрация по типу) |
| **Когда использовать** | Логирование, метрики | UI, интерактивное отображение |

### 5.2 Паттерн "Координатор"

Вернёмся к рассмотрению других архитектур **многоагентных систем**. Иногда удобно выстраивать управление как в реальной жизни - с помощью начальника, в роли которого выступает **координатор** — центральный хаб. Все агенты возвращают управление координатору. В этом случае не мы в явном виде прописываем цепочку передачи управления между агентами, а агент-координатор сам может выстроить цепочку передачи управления в зависимости от решаемой задачи.

```
        ┌──→ Planner ──┐
User → Coordinator ←──────┤
        ├──→ Searcher ─┤
        └──→ Writer ───┘
```

**Особенность:** Координатор решает, что делать дальше после каждого агента.

Обратите внимание, что в данном случае нам необходимы взаимные handoff-ы между агентами. Чтобы реализовать такие циклические зависимости, мы сначала описываем агент-координатор как заглушку, и потом уже наполняем его смыслом. 

In [ ]:
# Шаг 1: Создаём координатор-заглушку (для forward reference)
coordinator_agent = Agent(
    name="CoordinatorAgent",
    model=yandex_model,
    instructions="placeholder"  # Обновим позже
)

# Шаг 2: Создаём агентов с handoff ОБРАТНО к координатору
planner_agent = Agent(
    name="PlannerAgent",
    model=yandex_model,
    instructions="""
    Ты — планировщик исследований. Разбей тему на 3-5 ключевых вопросов.
    
    Формат ответа:
    ## План исследования
    1. [Вопрос]
    ...
    
    После создания плана ОБЯЗАТЕЛЬНО передай управление обратно CoordinatorHub.
    """,
    handoffs=[coordinator_agent]  # ← Возврат к координатору!
)

searcher_agent = Agent(
    name="SearcherAgent",
    model=yandex_model,
    instructions="""
    Ты — исследователь. Выполни поиск по полученным вопросам.
    Сохраняй ключевые факты с помощью save_note.
    
    После завершения поиска ОБЯЗАТЕЛЬНО передай управление обратно CoordinatorHub.
    """,
    tools=[WebSearchTool(), save_note],
    handoffs=[coordinator_agent]  # ← Возврат к координатору!
)

writer_agent = Agent(
    name="WriterAgent",
    model=yandex_model,
    instructions="""
    Ты — аналитик-писатель. Используй get_all_notes и напиши финальный отчёт.
    
    После написания отчёта передай управление обратно CoordinatorHub для финализации.
    Пиши на русском языке.
    """,
    tools=[get_all_notes],
    handoffs=[coordinator_agent]  # ← Возврат к координатору!
)

# Шаг 3: Обновляем координатор с полными handoffs
coordinator_agent.instructions = """
Ты — координатор исследовательской команды агентов.

Следуй этому процессу СТРОГО:
1. СНАЧАЛА передай задачу PlannerAgent для создания плана
2. Когда PlannerAgent вернёт план — передай его SearcherAgent для поиска
3. Когда SearcherAgent вернёт результаты — передай их WriterAgent для отчёта
4. Когда WriterAgent вернёт отчёт — верни ФИНАЛЬНЫЙ ответ пользователю

ВАЖНО: 
- Не отвечай пользователю напрямую до финального шага!
- Каждый раз когда к тебе возвращается управление, передай следующему агенту.
"""
coordinator_agent.handoffs = [planner_agent, searcher_agent, writer_agent]

print("✅ Паттерн \"координатор\": Coordinator ↔ Planner/Searcher/Writer")

В нашем примере в промтах мы явно прописывали, каким агентам в каких случаях следует передавать управление. Однако в зависимости от "умности" используемой LLM, она часто способна выстраивать разумные цепочки вызовов агентов без явных инструкций, исходя из смысла задачи.

Посмотрим, как такой подход работает. Будем использовать подход с Hooks как более простой, но можно использовать и streaming:

In [ ]:
research_notes = []  # Очищаем заметки

result = await Runner.run(
    coordinator_agent,
    "Исследуй тему: Тренды развития многоагентных систем в 2025 году",
    hooks=verbose_hooks,
    max_turns=20 # Необходимо увеличить из-за частых handoffs
)

print(f"Финальный агент: {result.last_agent.name}")
printx(result.final_output)

Поскольку мы использовали механизм handoff, то все агенты по-прежнему видят всю историю переписки.

### 5.3 Паттерн A: Agents-as-Tools (Агенты как инструменты)

В реальной жизни часто мы хотим делегировать задачу кому-то из специалистов, не вдаваясь в подробности, как они будут эту задачу решать. Для моделирования такой ситуации используется подход **агенты-как-инструменты**.

В этом случае вместо handoffs агенты вызывают других агентов с помощью механизма вызова инструментов. Агенты оборачиваются в инструменты с помощью функции `agent.as_tool()`. Координатор вызывает их как функции и получает результаты.

```
User → Coordinator ─┬─ call planner_tool() → result
                    ├─ call searcher_tool() → result
                    └─ call writer_tool() → result
                    → Final response
```

**Особенность:** Координатор полностью контролирует поток выполнения агентов. Sub-агенты не видят общую историю и, соответственно, не могут транслировать потоковые события - из-за чего подход на основе Streaming Events тут работать не будет. Да и отображение статусов через Hooks тоже работает только на вернхем уровне, до вызова под-агентов.

In [ ]:
# === ПАТТЕРН A: AGENTS-AS-TOOLS ===

# Sub-агенты (без handoffs — они будут вызываться как tools)
planner_tool_agent = Agent(
    name="PlannerTool",
    model=yandex_model,
    instructions="""
    Ты — планировщик. Разбей тему на 3-5 ключевых вопросов.
    Верни ТОЛЬКО план в виде нумерованного списка.
    """
)

searcher_tool_agent = Agent(
    name="SearcherTool",
    model=yandex_model,
    instructions="""
    Ты — исследователь. Получи вопросы и выполни поиск.
    Сохраняй факты с помощью save_note.
    Верни краткое резюме найденного.
    """,
    tools=[WebSearchTool(), save_note]
)

writer_tool_agent = Agent(
    name="WriterTool",
    model=yandex_model,
    instructions="""
    Ты — аналитик-писатель. Получи информацию и напиши связный отчёт.
    Используй get_all_notes для сбора заметок.
    Пиши на русском языке.
    """,
    tools=[get_all_notes]
)

# Координатор с агентами как инструментами
coordinator_tools = Agent(
    name="CoordinatorTools",
    model=yandex_model,
    instructions="""
    Ты — координатор исследовательской команды (Agents-as-Tools паттерн).
    
    Для исследования ПОСЛЕДОВАТЕЛЬНО вызови инструменты:
    1. planner_tool — получи план исследования
    2. searcher_tool — передай план, получи результаты поиска  
    3. writer_tool — передай результаты, получи финальный отчёт
    
    Верни пользователю финальный отчёт от writer_tool.
    """,
    tools=[
        planner_tool_agent.as_tool(
            tool_name="planner_tool",
            tool_description="Создаёт план исследования темы"
        ),
        searcher_tool_agent.as_tool(
            tool_name="searcher_tool",
            tool_description="Выполняет поиск по вопросам плана"
        ),
        writer_tool_agent.as_tool(
            tool_name="writer_tool",
            tool_description="Пишет финальный аналитический отчёт"
        )
    ]
)

print("✅ Паттерн A (Agents-as-Tools): Coordinator calls planner/searcher/writer as tools")

Посмотрим, как это работает:

In [ ]:
# Запуск Agents-as-Tools
research_notes = []  # Очищаем заметки

result_tools = await Runner.run(
    coordinator_tools,
    "Исследуй тему: Применение LLM для создания автономных агентов",
    hooks=verbose_hooks
)

print(f"Финальный агент: {result_tools.last_agent.name}")
printx(result_tools.final_output)

### Сравнение паттернов

Мы рассмотрели несколько паттернов:

| Аспект | Sequential Chain | Hub-and-Spoke | Agents-as-Tools |
|--------|------------------|---------------|------------------|
| **Кто отвечает** | Последний агент | Координатор | Координатор |
| **История разговора** | Полная передача | Полная передача | Новый контекст |
| **Гибкость** | Низкая (фиксированный путь) | Высокая (динамический роутинг) | Высокая |
| **Сложность отладки** | Средняя | Высокая | Низкая |
| **Токены** | Меньше (прямая цепочка) | Больше (много handoffs) | Больше (tool calls) |
| **Когда использовать** | Простые конвейеры | Динамические workflow | Сложная оркестрация |

Вообще паттернов междуагентного взаимодействия может быть много. Часто различают между [двумя типами многоагентных систем](https://www.anthropic.com/engineering/building-effective-agents):

* **Agentic Workflows** - система, в которой LLM оркестрируются с помощью программно предопределённого механизма. Наш первый пример (Sequential Chain) хорошо это демонстрирует.
* **Multi-Agent Systems** - в них процесс взаимодействия динамически определяется LLM.

На практике иногда бывает сложно различить эти два подхода, например, в наших примерах последовательность вызова агентов была прописана в промптах, хотя теоретически именно LLM решала, какой же агент вызвать.

---

## Заключение

Агентский подход позволяет решать более сложные задачи, разбивая их на отдельных агентов. При этом агенты являются в некотором смысле автономными, они могут сами планировать какие-то действия, вызывать необходимые инструменты и подстраваться под результаты их выполнения. Сообщества агентов могут использоваться для декомпозиции сложных задач и для построения систем управления на основе различных архитектур.